Movie Feature Engineering

In [58]:
import pandas as pd
import numpy as np
import pyarrow as pa
import re
from tqdm.notebook import tqdm

In [59]:
tqdm.pandas()

In [60]:
df = pd.read_csv("../data/movies.csv")
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,(2021),"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,(2021– ),"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,(2010–2022),"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,(2013– ),"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,(2021),"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9999 entries, 0 to 9998
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   MOVIES    9999 non-null   object 
 1   YEAR      9355 non-null   object 
 2   GENRE     9919 non-null   object 
 3   RATING    8179 non-null   float64
 4   ONE-LINE  9999 non-null   object 
 5   STARS     9999 non-null   object 
 6   VOTES     8179 non-null   object 
 7   RunTime   7041 non-null   float64
 8   Gross     460 non-null    object 
dtypes: float64(2), object(7)
memory usage: 703.2+ KB


## Data Cleaning

Year

In [62]:
df['YEAR'] = df['YEAR'].str.replace('(','',regex=False)
df['YEAR'] = df['YEAR'].str.replace(')','',regex=False)
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross
0,Blood Red Sky,2021,"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN
1,Masters of the Universe: Revelation,2021–,"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN
2,The Walking Dead,2010–2022,"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN
3,Rick and Morty,2013–,"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN
4,Army of Thieves,2021,"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN


In [63]:
df['TYPE'] = np.where(df['YEAR'].str.contains('–'), 'Series', 'Movie')
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE
0,Blood Red Sky,2021,"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN,Movie
1,Masters of the Universe: Revelation,2021–,"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN,Series
2,The Walking Dead,2010–2022,"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN,Series
3,Rick and Morty,2013–,"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN,Series
4,Army of Thieves,2021,"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN,Movie


In [64]:
df['YEAR_FROM'] = np.nan
df['YEAR_TO'] = np.nan

df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,YEAR_FROM,YEAR_TO
0,Blood Red Sky,2021,"\nAction, Horror, Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN,Movie,NaN,NaN
1,Masters of the Universe: Revelation,2021–,"\nAnimation, Action, Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN,Series,NaN,NaN
2,The Walking Dead,2010–2022,"\nDrama, Horror, Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN,Series,NaN,NaN
3,Rick and Morty,2013–,"\nAnimation, Adventure, Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN,Series,NaN,NaN
4,Army of Thieves,2021,"\nAction, Crime, Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN,Movie,NaN,NaN


In [65]:
df.sample(15)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,YEAR_FROM,YEAR_TO
5469,Chico Bon Bon and the Very Berry Holiday,2020 TV Movie,"\nAnimation, Family",5.8,\nWhen Barry is not shown to deliver the Blund...,\n Director:\nDarragh O'Connell\n| \n St...,32,25.0,NaN,Movie,NaN,NaN
1649,2gether,2020,"\nComedy, Drama, Romance",7.9,\nA student named Tine wants to get rid of a g...,\n \n Stars:\nVachirawit Chivaar...,"1,276",50.0,NaN,Movie,NaN,NaN
5790,#Rucker50,2016,\nDocumentary,5.2,\nAdd a Plot\n,\n Director:\nRobert McCullough Jr.\n| \n ...,105,56.0,NaN,Movie,NaN,NaN
8414,Clickbait,2021,"\nCrime, Drama, Mystery",NaN,\nAdd a Plot\n,\n \n Stars:\nElizabeth Alexande...,NaN,NaN,NaN,Movie,NaN,NaN
3540,Riding Faith,2020,\nFamily,4.4,\nWith the impending foreclose of the family R...,\n Director:\nPaco Aguilar\n| \n Stars:\...,196,81.0,NaN,Movie,NaN,NaN
5357,Hometown,NaN,\nComedy,NaN,\nPlot details kept under wraps.,\n \n Star:\nNatasha Rothwell\n,NaN,NaN,NaN,Series,NaN,NaN
6582,Bleach: Burîchi,2004–2012,"\nAnimation, Action, Adventure",8.0,\nPreparations are underway for Rukia's execut...,"\n Directors:\nNoriyuki Abe, \nYoshinori Od...",150,23.0,NaN,Series,NaN,NaN
8128,Night on Earth,2020,\nDocumentary,7.8,\nFrom the African savanna to be Peruvian dese...,\n Director:\nPeter Fison\n| \n Stars:\n...,303,53.0,NaN,Movie,NaN,NaN
9058,Undercover,2019–,"\nCrime, Drama, Thriller",7.2,\nBob sets up a meeting between the Berger bro...,\n Director:\nJoe Fria\n| \n Stars:\nTom...,236,50.0,NaN,Series,NaN,NaN
6893,Historia de un crimen: Colmenares,2019,\nCrime,6.9,"\nLaura and Jessy's bail hearing begins, trigg...",\n Director:\nFelipe Martínez Amador\n| \n ...,29,NaN,NaN,Movie,NaN,NaN


In [66]:
def extract_from(x):
    if pd.isna(x):
        return np.nan
    
    year_to_return = None

    year = str(x)
    if '–' not in year:
        year_to_return = year
    else:
        # 2010-2015 -> [2010, 2015]
        # 2010- -> [2010,]
        years = year.split('–')
        year_to_return = years[0]

    year_to_return = re.sub('[^0-9]', '', year_to_return)

    return year_to_return


def extract_to(x):
    if pd.isna(x):
        return np.nan
    
    year_to_return = None

    year = str(x)
    if '–' not in year:
        year_to_return = re.sub('[^0-9]', '', year)
        return year_to_return
    else:
        years = year.split('–')
        year_to_return = re.sub('[^0-9]', '', years[1])
        if len(year_to_return) == 0:
            return np.nan
        else:
            return year_to_return
    

df['YEAR_FROM'] = df['YEAR'].progress_apply(extract_from)
df['YEAR_TO'] = df['YEAR'].progress_apply(extract_to)

  0%|          | 0/9999 [00:00<?, ?it/s]

  0%|          | 0/9999 [00:00<?, ?it/s]

In [67]:
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,YEAR_FROM,YEAR_TO
5431,Bad Samaritans,2013,\nComedy,6.7,\nAn ex-couple accidentally burns down a chunk...,"\n \n Stars:\nBrian Kubach, \nJu...",432,30.0,NaN,Movie,2013,2013
1304,Ajeeb Daastaans,2021,"\nDrama, Romance",6.7,\nFour shorts explore the surprising ways in w...,"\n Directors:\nNeeraj Ghaywan, \nKayoze Ira...","4,649",142.0,NaN,Movie,2021,2021
6938,Brigada Costa del Sol,2019,"\nAction, Adventure, Drama",8.2,\nAdd a Plot\n,\n Director:\nMiguel Alcantud\n| \n Star...,12,NaN,NaN,Movie,2019,2019
1163,Heartbreak High,1994–1999,"\nDrama, Romance",7.8,\nThe series based on the lives of a group of ...,"\n \n Stars:\nCallan Mulvey, \nL...","2,870",50.0,NaN,Series,1994,1999
9557,Paradise PD,2018–,"\nAnimation, Action, Comedy",6.7,"\nDusty gives Fitz a new identity: ""Family Feu...",\n Director:\nLauren Andrews\n| \n Stars...,149,23.0,NaN,Series,2018,NaN
3513,The Bleeding Edge,2018,\nDocumentary,7.6,\nA look at the unforeseen consequences of adv...,\n Director:\nKirby Dick\n| \n Stars:\nR...,"2,374",99.0,NaN,Movie,2018,2018
8219,Country Comfort,2021,"\nComedy, Drama, Family",8.3,\nBeau and Summer's relationship takes center ...,\n Director:\nKelly Park\n| \n Stars:\nK...,78,NaN,NaN,Movie,2021,2021
8185,Next in Fashion,2020,\nReality-TV,7.5,\nGo big and bold - or go home? Prints polariz...,"\n \n Stars:\nAlexa Chung, \nTan...",98,NaN,NaN,Movie,2020,2020
3262,El faro de las orcas,2016,"\nBiography, Drama, Romance",6.8,\nA mother with an autistic child travel from ...,\n Director:\nGerardo Olivares\n| \n Sta...,"3,646",110.0,NaN,Movie,2016,2016
8076,Lupin,2021–,"\nAction, Crime, Drama",7.8,"\nAssane hatches a plot to contact Comet, an i...",\n Director:\nLouis Leterrier\n| \n Star...,"2,653",52.0,NaN,Series,2021,NaN


Genre

In [68]:
df['GENRE'] = df['GENRE'].str.replace(' ', '', regex=False)
df['GENRE'] = df['GENRE'].str.replace('\n', '', regex=False)
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,YEAR_FROM,YEAR_TO
5505,Magic for Humans by Mago Pop,2021–,"Comedy,Reality-TV",6.4,"\nIn this Spanish adaptation of ""Magic for Hum...","\n \n Stars:\nAntonio Díaz, \nCr...",59,NaN,NaN,Series,2021,NaN
2624,Grizzy and the Lemmings,2007–,"Animation,Comedy,Family",7.0,\nThe forest ranger's house is the only area o...,\n \n Stars:\nPierre-Alain de Ga...,445,7.0,NaN,Series,2007,NaN
3815,Hot Gimmick: Girl Meets Boy,2019,Drama,4.5,\nHatsumi and her family live normal lives unt...,\n Director:\nYûki Yamato\n| \n Stars:\n...,365,119.0,NaN,Movie,2019,2019
5936,Intimacy,NaN,Drama,NaN,\nA compromising sexual video featuring a prom...,"\n \n Stars:\nItziar Ituño, \nVe...",NaN,NaN,NaN,Series,NaN,NaN
4492,Shine Your Eyes,2020,"Drama,Thriller",6.6,\nA Nigerian musician travels to Brazil to fin...,\n Director:\nMatias Mariani\n| \n Stars...,276,102.0,NaN,Movie,2020,2020
5171,Betoben baireoseu,2008,"Comedy,Drama,Romance",7.6,\nKang Gun Woo is a world renowned orchestra m...,"\n \n Stars:\nMyung-Min Kim, \nJ...",318,NaN,NaN,Movie,2008,2008
9932,Ginny & Georgia,2021–,"Comedy,Drama",NaN,\nAdd a Plot\n,\n Director:\nAnya Adams\n| \n Stars:\nB...,NaN,NaN,NaN,Series,2021,NaN
5918,History's Greatest Hoaxes,2016–,Documentary,6.2,\nHistory's Greatest Hoaxes looks at some of t...,"\n \n Stars:\nTom Ward, \nGuy Wa...",36,NaN,NaN,Series,2016,NaN
9283,Las mágicas historias de Plim Plim,2011–2016,"Animation,Comedy,Family",NaN,\nAdd a Plot\n,"\n Directors:\nMikhail Plata, \nMichael Fue...",NaN,NaN,NaN,Series,2011,2016
8236,Kalifat,2020–,"Crime,Drama,Thriller",7.9,\nCalle visits with Suleiman and Tuba. Fatima ...,\n Director:\nGoran Kapetanovic\n| \n St...,278,47.0,NaN,Series,2020,NaN


In [69]:
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,YEAR_FROM,YEAR_TO
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN,Movie,2021,2021
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN,Series,2021,NaN
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN,Series,2010,2022
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN,Series,2013,NaN
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN,Movie,2021,2021


In [70]:
df_dummy = df['GENRE'].str.get_dummies(sep=',')
df_dummy = df_dummy.add_prefix('GENRE_')
df_dummy.head()

,GENRE_Action,GENRE_Adventure,GENRE_Animation,GENRE_Biography,GENRE_Comedy,GENRE_Crime,GENRE_Documentary,GENRE_Drama,GENRE_Family,GENRE_Fantasy,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
1,1,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,1,0,0
3,0,1,1,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [71]:
df_dummy.shape

(9999, 27)

In [72]:
df.shape

(9999, 12)

In [73]:
#Join/Merge the dummy dataframe with the original dataframe
#df = df.join(df_dummy)
df = df.merge(df_dummy, left_index=True, right_index=True)
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,\nA woman with a mysterious illness is forced ...,\n Director:\nPeter Thorwarth\n| \n Star...,"21,062",121.0,NaN,Movie,...,0,0,0,0,0,0,0,1,0,0
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,\nThe war for Eternia begins again in what may...,"\n \n Stars:\nChris Wood, \nSara...","17,870",25.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"\n \n Stars:\nAndrew Lincoln, \n...","885,805",44.0,NaN,Series,...,0,0,0,0,0,0,0,1,0,0
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,\nAn animated series that follows the exploits...,"\n \n Stars:\nJustin Roiland, \n...","414,849",23.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"\nA prequel, set before the events of Army of ...",\n Director:\nMatthias Schweighöfer\n| \n ...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0


Stars

In [74]:
df.sample(15)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
4348,Broken,2019–,Documentary,6.9,\nInfluencer hype and marketing create conditi...,"\n \n Stars:\nJenise Morgan, \nB...",959,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
7272,Teenage Bounty Hunters,2020,"Comedy,Crime,Drama",8.0,\nSterling and Blair can't escape relationship...,\n Director:\nLauren Morelli\n| \n Stars...,703,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
9664,The Talk,2010–,Talk-Show,NaN,\nSinger/songwriter H.E.R.; former NFL player/...,\n Director:\nJoseph Carolei\n| \n Stars...,NaN,38.0,NaN,Series,...,0,0,0,0,0,0,1,0,0,0
4352,Atari: Game Over,2014,Documentary,6.7,\nA crew digs up all of the old Atari 2600 gam...,\n Director:\nZak Penn\n| \n Stars:\nZak...,"5,762",66.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
1626,Bulbbul,2020,"Drama,Horror,Mystery",6.6,\nA man returns home after years to find his b...,\n Director:\nAnvita Dutt\n| \n Stars:\n...,"10,484",94.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3306,Two Lovers and a Bear,2016,"Adventure,Drama,Romance",6.1,\nSet in a small town near the North Pole wher...,\n Director:\nKim Nguyen\n| \n Stars:\nT...,"1,823",96.0,NaN,Movie,...,0,0,1,0,0,0,0,0,0,0
6536,Bleach: Burîchi,2004–2012,"Animation,Action,Adventure",8.7,\nCaptain Kurotsushi decides that rather than ...,"\n Directors:\nNoriyuki Abe, \nKazunori Miz...",181,23.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
3455,Intrusion,II 2021,Thriller,NaN,\nA woman moves to a small town with her husba...,\n Director:\nAdam Salky\n| \n Stars:\nL...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,1,0,0
8914,Jaguar,NaN,"Drama,History",NaN,\nAdd a Plot\n,"\n \n Stars:\nBlanca Suárez, \nI...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
1378,1899,2022–,"Drama,History,Horror",NaN,\nMultinational immigrants traveling from the ...,"\n \n Stars:\nAneurin Barnard, \...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0


In [75]:
df['STARS'] = df['STARS'].str.replace('\n', '', regex=False)
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,\nA woman with a mysterious illness is forced ...,Director:Peter Thorwarth| Stars:Peri B...,"21,062",121.0,NaN,Movie,...,0,0,0,0,0,0,0,1,0,0
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,\nThe war for Eternia begins again in what may...,"Stars:Chris Wood, Sarah Michel...","17,870",25.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"Stars:Andrew Lincoln, Norman R...","885,805",44.0,NaN,Series,...,0,0,0,0,0,0,0,1,0,0
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,\nAn animated series that follows the exploits...,"Stars:Justin Roiland, Chris Pa...","414,849",23.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"\nA prequel, set before the events of Army of ...",Director:Matthias Schweighöfer| Stars:...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0


In [76]:
df["Directors"] = None
df["Actors"] = None

def extract_directors(x):
    if "Director" in x:
        stars = x.split("|")
        if "Director" in stars[0]:
            return stars[0]
        else:
            return stars[1]
    else:
        return np.nan

def extract_actors(x):
    if "Star" in x:
        stars = x.split("|")
        if "Star" in stars[0]:
            return stars[0]
        else:
            return stars[1]
    else:
        return np.nan
    
df["Directors"] = df["STARS"].progress_apply(extract_directors)
df["Actors"] = df["STARS"].progress_apply(extract_actors)

df.head()


  0%|          | 0/9999 [00:00<?, ?it/s]

  0%|          | 0/9999 [00:00<?, ?it/s]

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western,Directors,Actors
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,\nA woman with a mysterious illness is forced ...,Director:Peter Thorwarth| Stars:Peri B...,"21,062",121.0,NaN,Movie,...,0,0,0,0,0,1,0,0,Director:Peter Thorwarth,"Stars:Peri Baumeister, Carl Anton Koch, A..."
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,\nThe war for Eternia begins again in what may...,"Stars:Chris Wood, Sarah Michel...","17,870",25.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Stars:Chris Wood, Sarah Michel..."
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"Stars:Andrew Lincoln, Norman R...","885,805",44.0,NaN,Series,...,0,0,0,0,0,1,0,0,NaN,"Stars:Andrew Lincoln, Norman R..."
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,\nAn animated series that follows the exploits...,"Stars:Justin Roiland, Chris Pa...","414,849",23.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Stars:Justin Roiland, Chris Pa..."
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"\nA prequel, set before the events of Army of ...",Director:Matthias Schweighöfer| Stars:...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,Director:Matthias Schweighöfer,"Stars:Matthias Schweighöfer, Nathalie Emm..."


In [77]:
df.sample(15)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western,Directors,Actors
7110,Supergirl,2015–2021,"Action,Adventure,Drama",6.3,\nNia's roommate is attacked by a man targetin...,Director:Armen V. Kevorkian| Stars:Mel...,"1,209",42.0,NaN,Series,...,0,0,0,0,0,0,0,0,Director:Armen V. Kevorkian,"Stars:Melissa Benoist, Chyler Leigh, Kati..."
3720,Plastic Cup Boyz: Laughing My Mask Off!,2021 TV Special,Comedy,4.7,\nA collection of stand-up specials that spotl...,Director:Royale Watkins| Stars:Will 'S...,10,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,Director:Royale Watkins,"Stars:Will 'Spank' Horton, Na'im Lynn, Jo..."
3422,Hate Story,2012,"Drama,Thriller",5.3,\nAfter a powerful businessman has her baby fo...,Director:Vivek Agnihotri| Stars:Nikhil...,"3,524",140.0,NaN,Movie,...,0,0,0,0,0,1,0,0,Director:Vivek Agnihotri,"Stars:Nikhil Dwivedi, Gulshan Devaiah, Pa..."
4321,Para Sempre Chape,2018,Documentary,7.2,\nFOREVER CHAPE tells the story of the soccer ...,Director:Luis Ara,298,80.0,NaN,Movie,...,0,0,0,0,0,0,0,0,Director:Luis Ara,NaN
3923,Kulipari: An Army of Frogs,2016,"Animation,Action,Fantasy",7.0,\nAnthropomorphic frogs battle the forces of d...,"Stars:Charlie Adler, Lacey Cha...",235,20.0,NaN,Movie,...,0,0,0,0,0,0,0,0,NaN,"Stars:Charlie Adler, Lacey Cha..."
8926,High Score,2020,"Documentary,History",7.4,\nSega's Genesis console and its speedy charac...,"Directors:William Acks, France Costrel, Sa...",420,41.0,NaN,Movie,...,0,0,0,0,0,0,0,0,"Directors:William Acks, France Costrel, Sa...","Stars:Charles Martinet, Tom Kalinske, Hir..."
9737,Inside Job,2021–,"Animation,Comedy",NaN,\nAdd a Plot\n,"Stars:Will Blagrove, Tisha Cam...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Stars:Will Blagrove, Tisha Cam..."
2086,Rugal,2020–,"Action,Crime,Sci-Fi",6.4,"\nBased off the webtoon of the same name, it's...","Stars:Choi Jin-Hyuk, Dong-Hyuk...",672,60.0,NaN,Series,...,0,1,0,0,0,0,0,0,NaN,"Stars:Choi Jin-Hyuk, Dong-Hyuk..."
3195,The Big Flower Fight,2020–,Reality-TV,7.4,"\nTen pairs of florists, sculptors and garden ...",Stars:James Alexander-Sinclair...,964,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,Stars:James Alexander-Sinclair...
8154,Glitch Techs,2020–,"Animation,Action,Adventure",8.7,"\nWhen an ultra-rare, ultra-dangerous glitch t...","Directors:Jose Romar Escolastico, Julius M...",40,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,"Directors:Jose Romar Escolastico, Julius M...","Stars:Ricardo Hurtado, Monica Ray, Luke Y..."


In [78]:
df["Directors"] = df["Directors"].str.replace('Director:', '', regex=False)
df["Directors"] = df["Directors"].str.replace('Directors:', '', regex=False)
df["Actors"] = df["Actors"].str.replace('Star:', '', regex=False)
df["Actors"] = df["Actors"].str.replace('Stars:', '', regex=False)
df.head(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western,Directors,Actors
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,\nA woman with a mysterious illness is forced ...,Director:Peter Thorwarth| Stars:Peri B...,"21,062",121.0,NaN,Movie,...,0,0,0,0,0,1,0,0,Peter Thorwarth,"Peri Baumeister, Carl Anton Koch, Alexand..."
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,\nThe war for Eternia begins again in what may...,"Stars:Chris Wood, Sarah Michel...","17,870",25.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Chris Wood, Sarah Michelle Gel..."
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,\nSheriff Deputy Rick Grimes wakes up from a c...,"Stars:Andrew Lincoln, Norman R...","885,805",44.0,NaN,Series,...,0,0,0,0,0,1,0,0,NaN,"Andrew Lincoln, Norman Reedus,..."
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,\nAn animated series that follows the exploits...,"Stars:Justin Roiland, Chris Pa...","414,849",23.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Justin Roiland, Chris Parnell,..."
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"\nA prequel, set before the events of Army of ...",Director:Matthias Schweighöfer| Stars:...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,Matthias Schweighöfer,"Matthias Schweighöfer, Nathalie Emmanuel,..."
5,Outer Banks,2020–,"Action,Crime,Drama",7.6,\nA group of teenagers from the wrong side of ...,"Stars:Chase Stokes, Madelyn Cl...","25,858",50.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Chase Stokes, Madelyn Cline, M..."
6,The Last Letter from Your Lover,2021,"Drama,Romance",6.8,\nA pair of interwoven stories set in the past...,Director:Augustine Frizzell| Stars:Sha...,"5,283",110.0,NaN,Movie,...,1,0,0,0,0,0,0,0,Augustine Frizzell,"Shailene Woodley, Joe Alwyn, Wendy Nottin..."
7,Dexter,2006–2013,"Crime,Drama,Mystery",8.6,"\nBy day, mild-mannered Dexter is a blood-spat...","Stars:Michael C. Hall, Jennife...","665,387",53.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Michael C. Hall, Jennifer Carp..."
8,Never Have I Ever,2020–,Comedy,7.9,\nThe complicated life of a modern-day first g...,"Stars:Maitreyi Ramakrishnan, P...","34,530",30.0,NaN,Series,...,0,0,0,0,0,0,0,0,NaN,"Maitreyi Ramakrishnan, Poorna ..."
9,Virgin River,2019–,"Drama,Romance",7.4,"\nSeeking a fresh start, nurse practitioner Me...","Stars:Alexandra Breckenridge, ...","27,279",44.0,NaN,Series,...,1,0,0,0,0,0,0,0,NaN,"Alexandra Breckenridge, Martin..."


Dummy on Actors

In [79]:
df["Actors"] = df["Actors"].str.replace(", ", ",", regex=False)
df["Actors"] = df["Actors"].str.strip()

df_dummy = df['Actors'].str.get_dummies(sep=',')
df_dummy = df_dummy.add_prefix('Actor_')
df_dummy.head()

,Actor_2 Chainz,Actor_2'Live Bre,Actor_2Mex,Actor_50 Cent,Actor_A Boogie wit da Hoodie,Actor_A.J. Baime,Actor_A.J. Daulerio,Actor_A.J. LoCascio,Actor_A.N.T.I.,Actor_AJ Bowen,...,Actor_Özge Borak,Actor_Özge Özpirinçci,Actor_Özgür Emre Yildirim,Actor_Özgür Ozan,Actor_Özkan Ugur,Actor_Özz Nûjen,Actor_Úrsula Corberó,Actor_Úrsula Pruneda,Actor_Ülkü Duru,Actor_Þorsteinn Bachmann
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [80]:
df_dummy.shape

(9999, 17325)

In [81]:
#merge/join the dummy dataframe with the original dataframe
df = df.merge(df_dummy, left_index=True, right_index=True)
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Actor_Özge Borak,Actor_Özge Özpirinçci,Actor_Özgür Emre Yildirim,Actor_Özgür Ozan,Actor_Özkan Ugur,Actor_Özz Nûjen,Actor_Úrsula Corberó,Actor_Úrsula Pruneda,Actor_Ülkü Duru,Actor_Þorsteinn Bachmann
5411,Doomsdays,2013,Drama,6.7,\nDirty Fred (Justin Rice) and Bruho (Leo Fitz...,Director:Eddie Mullins| Stars:Leo Fitz...,533,91.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
5542,God Loves Uganda,2013,Documentary,7.4,\nAn account of the American Evangelicals' att...,Director:Roger Ross Williams,"1,179",83.0,$0.05M,Movie,...,0,0,0,0,0,0,0,0,0,0
1342,Miss Bala,2019,"Action,Crime,Drama",5.8,\nGloria finds a power she never knew she had ...,Director:Catherine Hardwicke| Stars:Gi...,"10,048",104.0,$15.01M,Movie,...,0,0,0,0,0,0,0,0,0,0
61,Legends of Tomorrow,2016–,"Action,Adventure,Drama",6.8,\nTime-travelling rogue Rip Hunter has to recr...,"Stars:Caity Lotz, Amy Louise P...","95,897",42.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4580,Tell Me Everything,NaN,Drama,NaN,\nSix close students in a small New England to...,Director:Leslye Headland,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
9856,Surviving Summer,2022–,Drama,NaN,\nAdd a Plot\n,"Director:Ben Chessell| Stars:Sky Katz,...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
2057,Pokémon: Mewtwo Strikes Back - Evolution,2019,"Animation,Action,Family",5.7,\nAfter a scientific experiment leads to the c...,"Directors:Motonori Sakakibara, Tetsuo Yaji...","4,870",98.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
6108,The Deep Mad Dark,2018 TV Movie,Mystery,NaN,\nAdd a Plot\n,Director:Niels Arden Oplev,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
8246,The Late Late Show with James Corden,2015–,"Comedy,Talk-Show",NaN,"\nActor Patrick Stewart (""Star Trek: Picard"");...","Stars:James Corden, Reggie Wat...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4712,Revenge of the Pontianak,2019,"Horror,Romance",5.2,"\n1965, Malaysia. A small village helps Khalid...","Directors:Glen Goei, Gavin Yap| Stars:...",248,92.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0


Dummy on Director

In [82]:
df["Directors"] = df["Directors"].str.replace(", ", ",", regex=False)
df["Directors"] = df["Directors"].str.strip()

df_dummy = df['Directors'].str.get_dummies(sep=',')
df_dummy = df_dummy.add_prefix('Director_')
df_dummy.head()

,Director_Aadish Keluskar,Director_Aaron Augenblick,Director_Aaron Burns,Director_Aaron Hann,Director_Aaron Lieber,Director_Aaron Long,Director_Aaron Moorhead,Director_Aaron Saidman,Director_Aaron Sorkin,Director_Aban Raza,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [83]:
#merge the dummy dataframe with the original dataframe
df = df.merge(df_dummy, left_index=True, right_index=True)
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
5342,Sea People,1999 TV Movie,"Drama,Family,Fantasy",5.9,"\nHume Cronyn, Joan Gregson and Tegan Moss sta...","Director:Vic Sarin| Stars:Hume Cronyn,...",438,95.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
7758,The Politician,2019–,"Comedy,Drama",8.5,"\nWith the senate race up for grabs, Payton an...",Director:Gwyneth Horder-Payton| Stars:...,476,44.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
2436,Zbrodnia,2014–,"Action,Crime",6.2,\nThe peaceful lives of the residents inhabiti...,"Stars:Magdalena Boczarska, Woj...",558,60.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4341,The 43,2019–,Documentary,7.2,\nThis docuseries disputes the Mexican governm...,Star:Paco Ignacio Taibo II,99,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
7175,Go! Vive a Tu Manera,2019,"Comedy,Musical,Romance",8.6,\nZoe faces off against Lupe about her behavio...,"Stars:Darcy Rose Byrnes, Rebec...",9,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3540,Riding Faith,2020,Family,4.4,\nWith the impending foreclose of the family R...,Director:Paco Aguilar| Stars:John Schn...,196,81.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
976,Don't F**k with Cats: Hunting an Internet Killer,2019,"Documentary,Crime",8.0,\nA group of online justice seekers track down...,"Stars:Deanna Thompson, John Gr...","42,827",187.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
2454,Truth or Dare,II 2012,"Horror,Mystery",5.6,\nYoung British boys and girls travel to an is...,"Director:Robert Heath| Stars:Tom Kane,...","8,980",96.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
9399,First Kill,NaN,"Drama,Horror,Mystery",NaN,\nAdd a Plot\n,Director:Jet Wilkinson| Stars:Elizabet...,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
8765,Dark,2017–2020,"Crime,Drama,Mystery",9.3,\nAdam holds Martha captive in 2020. On the da...,Director:Baran bo Odar| Stars:Lisa Vic...,"11,525",59.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0


In [84]:
df.shape

(9999, 21449)

In [85]:
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
6911,Cobra Kai,2018–,"Action,Comedy,Drama",8.0,\nDaniel's tarnished public image takes a toll...,Director:Lin Oeding| Stars:Ralph Macch...,"2,540",32.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4333,The Oscars,2017 TV Special,Music,6.7,\nThe 89th Annual Academy Awards ceremony cele...,Director:Glenn Weiss| Stars:Jimmy Kimm...,"1,901",229.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
5318,Joe Cocker: Mad Dog with Soul,2017,Documentary,7.2,\nThe story of singer Joe Cocker is told throu...,Director:John Edginton| Stars:Joe Cock...,772,90.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
8531,Too Hot to Handle,2020–,"Game-Show,Reality-TV,Romance",4.8,"\nAfter another pricey peck, the group dynamic...","Stars:Bryce Hirschberg, Chloe ...",262,43.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
7145,Grand Army,2020,Drama,7.2,\nJoey stages a protest at school. Owen and Ja...,Director:So Yong Kim| Stars:Odessa A’z...,126,49.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3305,The Secrets of Emily Blair,2016,"Horror,Thriller",4.1,\nDesperate to save his fiancee from a demon t...,Director:Joseph P. Genier| Stars:Ellen...,"1,332",95.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
4612,Babies Behind Bars,2011 TV Movie,Documentary,6.2,"\nFollows pregnancy and birth in prison, inclu...",Director:Amanda Richardson| Stars:Lesl...,110,120.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
7800,The Dragon Prince,2018–,"Animation,Adventure,Drama",8.8,\nRayla and Callum race across the desert to f...,Director:Villads Spangsberg| Stars:Rac...,447,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
390,Lucy,I 2014,"Action,Sci-Fi,Thriller",6.4,"\nA woman, accidentally caught in a dark deal,...",Director:Luc Besson| Stars:Scarlett Jo...,"462,122",89.0,$126.66M,Movie,...,0,0,0,0,0,0,0,0,0,0
337,She-Ra and the Princesses of Power,2018–2020,"Animation,Action,Adventure",7.9,"\nShe-Ra, Princess of Power, leads a rebellion...","Stars:Aimee Carrero, Marcus Sc...","11,967",30.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0


One-Line

In [86]:
df["ONE-LINE"] = df["ONE-LINE"].str.replace('\n', '', regex=False)
df["ONE-LINE"] = np.where(df["ONE-LINE"] == "Add a Plot", np.nan, df["ONE-LINE"])
df.sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
2207,The Millionaire Detective: Balance - Unlimited,2020,"Animation,Crime,Drama",7.4,A rich but eccentric detective and a middle-cl...,"Stars:Mamoru Miyano, Yûsuke Oh...","1,105",23.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
8516,Valeria,2020–,"Comedy,Drama,Romance",7.5,NaN,Director:Nely Reguera| Stars:Diana Góm...,68,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
394,Wet Hot American Summer: Ten Years Later,2017,Comedy,6.9,The campers and counselors of Camp Firewood me...,"Stars:Nina Hellman, Marguerite...","7,784",216.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3850,Di-eo Ma-i Peu-ren-jeu,2016–,"Comedy,Drama",8.3,Depict the life story of the ones in their twi...,"Stars:Hyun-Jung Go, Jung-Hyun ...",262,70.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
9116,Vikings: Valhalla,NaN,"Action,Adventure,Drama",NaN,NaN,"Stars:Laura Berlin, Karen Conn...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
1508,Into the Forest,2015,"Drama,Thriller",5.8,"After a massive power outage, two sisters lear...",Director:Patricia Rozema| Stars:Elliot...,"19,853",101.0,$0.01M,Movie,...,0,0,0,0,0,0,0,0,0,0
3287,Bake Squad,2021–,"Game-Show,Reality-TV",NaN,Expert bakers elevate desserts with next-level...,Star:Christina Tosi,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
9618,Dad Stop Embarrassing Me,2021,"Comedy,Family",6.0,Brian receives an unexpected business offer. P...,Director:Bentley Kyle Evans| Stars:Jam...,49,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
7248,Midnight Mass,2021–,"Drama,Horror,Mystery",NaN,NaN,Director:Mike Flanagan| Stars:Annabeth...,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
6400,Meeting a Bullet,2004 Video,Crime,3.8,A dirty cop accuses a young individual of a st...,Director:Douglas Elford-Argent| Stars:...,115,75.0,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0


## Missing Data

In [87]:
columns_to_check = ['ONE-LINE', 'MOVIES', 'RATING', 'VOTES', 
                    'RunTime', 'Gross', 'TYPE', 'YEAR_FROM', 'YEAR_TO','GENRE']

missing_df = df[columns_to_check].isnull().sum().to_frame()
missing_df = missing_df.rename(columns={0: 'Missing Count'})
missing_df['missing_percentage'] = (missing_df['Missing Count'] / len(df)) * 100 #shape[0]
missing_df

,Missing Count,missing_percentage
ONE-LINE,1265,12.651265
MOVIES,0,0.000000
RATING,1820,18.201820
VOTES,1820,18.201820
RunTime,2958,29.582958
Gross,9539,95.399540
TYPE,0,0.000000
YEAR_FROM,644,6.440644
YEAR_TO,3824,38.243824
GENRE,80,0.800080


YEAR_TO

In [88]:
df['YEAR_TO'].dtypes

dtype('O')

In [89]:
#Take the current year
current_year = pd.Timestamp.now().year

df["YEAR_TO"] = np.where(
    df["YEAR_FROM"].notna() & df["YEAR_TO"].isna(),
    current_year,
    df["YEAR_TO"]
)

In [90]:
missing_df = df[columns_to_check].isnull().sum().to_frame()
missing_df

,0
ONE-LINE,1265
MOVIES,0
RATING,1820
VOTES,1820
RunTime,2958
Gross,9539
TYPE,0
YEAR_FROM,644
YEAR_TO,644
GENRE,80


Run-Time

In [91]:
df["RunTime"].describe()

count    7041.000000
mean       68.688539
std        47.258056
min         1.000000
25%        36.000000
50%        60.000000
75%        95.000000
max       853.000000
Name: RunTime, dtype: float64

In [92]:
df[df['RunTime'] >= 600]

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
1081,Soupçons,2004–2018,"Documentary,Crime,Drama",7.9,The high-profile murder trial of American nove...,"Stars:Michael Peterson, David ...","20,200",629.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
1902,El tiempo entre costuras,2013–2014,"Adventure,Drama,History",8.3,Sira Quiroga is a young Spanish dressmaker eng...,"Stars:Adriana Ugarte, Mari Car...","3,876",853.0,NaN,Series,...,0,0,0,0,0,0,0,0,0,0


In [93]:
df[df["RunTime"].isna()].sample(10)

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
1623,Leave the World Behind,NaN,Drama,NaN,Family drama based on the upcoming novel by Ru...,Director:Sam Esmail| Stars:Denzel Wash...,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
3216,Most Dangerous Game,NaN,Comedy,NaN,Plot details are under wraps.,"Stars:Zoey Deutch, Glen Powell...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
1986,The Last Kids on Earth,2019–,"Animation,Adventure,Comedy",7.4,Young teenager Jack Sullivan and a group of fr...,"Stars:Nick Wolfhard, Montse He...","1,034",NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
5103,Uprising,VI,"Action,Thriller",NaN,After a global viral outbreak that turns peopl...,Director:Travis Knight,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
756,The School for Good and Evil,2022,"Action,Drama,Fantasy",NaN,A group of boys and girls are taken to an inst...,Director:Paul Feig| Stars:Charlize The...,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3880,Baahubali: Before the Beginning,NaN,"Action,Adventure,Drama",NaN,A mutinous and bitter girl with a vengeance ag...,"Stars:Snigdha Akolkar, Bijay A...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
6176,It's Wednesday Night,TV Special,Comedy,NaN,"With their exes watching the kids, a group of ...",,NaN,NaN,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
3934,Untitled Vince Vaughn/Netflix Project,NaN,"Action,Comedy",NaN,Plot kept under wraps.,Star:Vince Vaughn,NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
6986,La Reina de Indias y el Conquistador,2020–,"Drama,History",NaN,When the governor orders Catalina's return to ...,"Stars:Emmanuel Esparza, Essine...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
9090,Vikings: Valhalla,NaN,"Action,Adventure,Drama",NaN,NaN,"Stars:James Ballanger, Laura B...",NaN,NaN,NaN,Series,...,0,0,0,0,0,0,0,0,0,0


In [94]:
df["RunTime"][df["TYPE"] == "Series"].describe()

count    2904.000000
mean       39.368802
std        29.549575
min         1.000000
25%        24.000000
50%        38.000000
75%        47.000000
max       853.000000
Name: RunTime, dtype: float64

In [95]:
df["RunTime"][df["TYPE"] == "Movie"].describe()

count    4137.000000
mean       89.269761
std        46.489358
min         1.000000
25%        64.000000
50%        90.000000
75%       105.000000
max       573.000000
Name: RunTime, dtype: float64

In [96]:
movie_mean_runtime = df.loc[df["TYPE"] == "Movie", "RunTime"].mean()
df.loc[(df["TYPE"] == "Movie")&(df["RunTime"].isna()), "RunTime"] = movie_mean_runtime

In [97]:
movie_mean_runtime

np.float64(89.26976069615664)

In [98]:
df["RunTime"][df["TYPE"] == "Movie"].isna().sum()

np.int64(0)

In [99]:
series_mean_runtime = df.loc[df["TYPE"] == "Series", "RunTime"].mean()
series_mean_runtime

np.float64(39.368801652892564)

In [100]:
df.loc[(df["TYPE"] == "Series")&(df["RunTime"].isna()), "RunTime"] = series_mean_runtime

In [101]:
df["RunTime"][df["TYPE"] == "Series"].isna().sum()

np.int64(0)

In [102]:
df["VOTES"] = df["VOTES"].str.replace(",", "", regex=False)
df.head()

,MOVIES,YEAR,GENRE,RATING,ONE-LINE,STARS,VOTES,RunTime,Gross,TYPE,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
0,Blood Red Sky,2021,"Action,Horror,Thriller",6.1,A woman with a mysterious illness is forced in...,Director:Peter Thorwarth| Stars:Peri B...,21062,121.000000,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0
1,Masters of the Universe: Revelation,2021–,"Animation,Action,Adventure",5.0,The war for Eternia begins again in what may b...,"Stars:Chris Wood, Sarah Michel...",17870,25.000000,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
2,The Walking Dead,2010–2022,"Drama,Horror,Thriller",8.2,Sheriff Deputy Rick Grimes wakes up from a com...,"Stars:Andrew Lincoln, Norman R...",885805,44.000000,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
3,Rick and Morty,2013–,"Animation,Adventure,Comedy",9.2,An animated series that follows the exploits o...,"Stars:Justin Roiland, Chris Pa...",414849,23.000000,NaN,Series,...,0,0,0,0,0,0,0,0,0,0
4,Army of Thieves,2021,"Action,Crime,Horror",NaN,"A prequel, set before the events of Army of th...",Director:Matthias Schweighöfer| Stars:...,NaN,89.269761,NaN,Movie,...,0,0,0,0,0,0,0,0,0,0


In [103]:
df = df.dropna(subset=["GENRE", "RATING", "YEAR_FROM", "YEAR_TO"])

In [104]:
column_names = ["MOVIES", "GENRE", "RATING", "ONE-LINE",
                "STARS", "VOTES", "RunTime", "Gross", "TYPE",
                "YEAR_FROM", "YEAR_TO"]
missing_df = df[column_names].isna().sum().to_frame()
missing_df = missing_df.rename(columns={0:"missing"})
missing_df["percentage"] = (missing_df["missing"] / df.shape[0]) * 100
missing_df

,missing,percentage
MOVIES,0,0.000000
GENRE,0,0.000000
RATING,0,0.000000
ONE-LINE,371,4.542116
STARS,0,0.000000
VOTES,0,0.000000
RunTime,0,0.000000
Gross,7708,94.368266
TYPE,0,0.000000
YEAR_FROM,0,0.000000


In [105]:
df.drop(columns=["MOVIES", "YEAR", "GENRE", "STARS", "ONE-LINE", "Gross",
                 "Directors", "Actors"], inplace=True)

In [106]:
df.dtypes

RATING                            float64
VOTES                              object
RunTime                           float64
TYPE                               object
YEAR_FROM                          object
                                   ...   
Director_Ángel Gómez Hernández      int64
Director_Ángeles Reiné              int64
Director_Åke Sandgren               int64
Director_Óscar Pedraza              int64
Director_Ömer Ugur                  int64
Length: 21441, dtype: object

# Visualization

In [107]:
import plotly.express as px

In [108]:
fig = px.histogram(df, x="RATING", y="VOTES", color="TYPE", barmode="group",
                   title="Votes by Rating and Type", labels={"RATING": "Rating", "VOTES": "Votes"})
fig.show()

In [109]:
fig = px.scatter_3d(df, x="RATING", y="VOTES", z="RunTime", color="TYPE", opacity=0.7)
fig.show()

In [110]:
#boxplot of runtime by type
fig = px.box(df, x="TYPE", y="RunTime", color="TYPE",
             title="Runtime by Type", labels={"TYPE": "Type", "RunTime": "Runtime"})
fig.show()

In [111]:
#Violin plot of rating by type
fig = px.violin(df, y=["RATING"], color="TYPE", points="all", box=True)
fig.show()

In [112]:
fig = px.histogram(df, x="VOTES", color="TYPE")
fig.show()

In [113]:
# Violin plot of votes by type
fig = px.violin(df, y=["VOTES"], color="TYPE", points="all", box=True)
fig.show()


In [ ]:
df_test = df[(df["Actor_Brad Pitt"] == 1) | (df["Actor_Angelina Jolie"] == 1)]
df_test

,RATING,VOTES,RunTime,TYPE,YEAR_FROM,YEAR_TO,GENRE_Action,GENRE_Adventure,GENRE_Animation,GENRE_Biography,...,Director_Àlex Pastor,Director_Álex de la Iglesia,Director_Álvaro Brechner,Director_Álvaro Fernández Armero,Director_Álvaro Longoria,Director_Ángel Gómez Hernández,Director_Ángeles Reiné,Director_Åke Sandgren,Director_Óscar Pedraza,Director_Ömer Ugur
300,7.6,386263,133.0,Movie,2011,2011,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
465,7.1,141657,95.0,Movie,2016,2016,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
615,6.0,230565,103.0,Movie,2010,2010,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
994,6.2,62161,111.0,Movie,1997,1997,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1045,6.0,43574,122.0,Movie,2017,2017,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Finding Genres with highest rating

In [ ]:
columns = df.columns.tolist()
genres = [col for col in columns if col.startswith("GENRE_")]
genres


['GENRE_Action',
 'GENRE_Adventure',
 'GENRE_Animation',
 'GENRE_Biography',
 'GENRE_Comedy',
 'GENRE_Crime',
 'GENRE_Documentary',
 'GENRE_Drama',
 'GENRE_Family',
 'GENRE_Fantasy',
 'GENRE_Film-Noir',
 'GENRE_Game-Show',
 'GENRE_History',
 'GENRE_Horror',
 'GENRE_Music',
 'GENRE_Musical',
 'GENRE_Mystery',
 'GENRE_News',
 'GENRE_Reality-TV',
 'GENRE_Romance',
 'GENRE_Sci-Fi',
 'GENRE_Short',
 'GENRE_Sport',
 'GENRE_Talk-Show',
 'GENRE_Thriller',
 'GENRE_War',
 'GENRE_Western']

In [ ]:
genre_rating = {}
for genre in genres:
    ratings = df[df[genre] == 1]["RATING"]
    genre_rating[genre] = ratings

df_genre = pd.DataFrame.from_dict(genre_rating)
df_genre.head()

,GENRE_Action,GENRE_Adventure,GENRE_Animation,GENRE_Biography,GENRE_Comedy,GENRE_Crime,GENRE_Documentary,GENRE_Drama,GENRE_Family,GENRE_Fantasy,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
0,6.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.1,NaN,NaN
1,5.0,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.2,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.2,NaN,NaN
3,NaN,9.2,9.2,NaN,9.2,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,7.6,NaN,NaN,NaN,NaN,7.6,NaN,7.6,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_genre.describe()

,GENRE_Action,GENRE_Adventure,GENRE_Animation,GENRE_Biography,GENRE_Comedy,GENRE_Crime,GENRE_Documentary,GENRE_Drama,GENRE_Family,GENRE_Fantasy,...,GENRE_News,GENRE_Reality-TV,GENRE_Romance,GENRE_Sci-Fi,GENRE_Short,GENRE_Sport,GENRE_Talk-Show,GENRE_Thriller,GENRE_War,GENRE_Western
count,1844.000000,1337.000000,1405.000000,268.000000,2419.000000,1377.000000,1137.000000,3507.000000,371.000000,456.000000,...,18.000000,348.000000,764.000000,270.000000,180.000000,160.000000,23.00000,777.000000,45.000000,20.000000
mean,7.097451,7.298504,7.379929,7.054104,6.825589,7.080320,7.174406,7.093413,6.774933,6.999123,...,7.061111,6.626437,6.802094,6.582593,6.772778,6.852500,6.96087,6.333719,6.986667,6.720000
std,1.285180,1.211768,1.113301,0.846153,1.236879,1.119748,0.871702,1.111281,1.262962,1.196625,...,0.991220,1.270726,1.115221,1.220957,1.142761,1.221451,1.63672,1.331206,1.063784,0.759917
min,2.000000,2.100000,3.100000,2.900000,1.100000,2.100000,2.500000,1.100000,2.800000,2.100000,...,5.000000,2.500000,3.000000,1.800000,3.200000,1.100000,3.60000,2.000000,3.200000,5.700000
25%,6.400000,6.600000,6.700000,6.600000,6.000000,6.500000,6.700000,6.500000,5.800000,6.400000,...,6.450000,5.800000,6.100000,6.000000,6.300000,6.200000,6.10000,5.500000,6.500000,6.275000
50%,7.200000,7.400000,7.500000,7.100000,7.000000,7.200000,7.300000,7.200000,6.700000,7.200000,...,7.250000,6.900000,7.000000,6.800000,6.800000,7.150000,7.30000,6.600000,7.200000,6.500000
75%,8.000000,8.200000,8.200000,7.600000,7.800000,7.800000,7.700000,7.900000,7.800000,7.800000,...,7.800000,7.500000,7.700000,7.400000,7.400000,7.600000,7.85000,7.300000,7.600000,7.050000
max,9.900000,9.900000,9.900000,9.100000,9.900000,9.800000,9.600000,9.900000,9.600000,9.500000,...,8.300000,9.200000,8.900000,9.200000,9.400000,9.400000,9.40000,9.400000,8.700000,8.300000


In [116]:
df.dtypes

RATING                            float64
VOTES                              object
RunTime                           float64
TYPE                               object
YEAR_FROM                          object
                                   ...   
Director_Ángel Gómez Hernández      int64
Director_Ángeles Reiné              int64
Director_Åke Sandgren               int64
Director_Óscar Pedraza              int64
Director_Ömer Ugur                  int64
Length: 21441, dtype: object

Save data in Parquet

In [ ]:
#!pip install pandas pyarrow

df["YEAR_FROM"] = pd.to_numeric(df["YEAR_FROM"], errors="coerce")
df["YEAR_TO"] = pd.to_numeric(df["YEAR_TO"], errors="coerce")
df["VOTES"] = pd.to_numeric(df["VOTES"], errors="coerce")

df.to_parquet("../data/movies_cleaned.parquet", engine='pyarrow')

In [121]:
missing_df = df["VOTES"].isna().sum()
missing_df

np.int64(0)